# Example: Calculation of reflectance images from radiance images in MASSIMAL dataset
This notebook is a demonstration of using the [MassiPipe](https://github.com/mh-skjelvareid/massipipe) data processing pipeline to create hyperspectral reflectance images from radiance images and downwelling irradiance from the [MASSIMAL dataset](https://doi.org/10.11582/2025.00041). The example uses data from the ZIP file [massimal_vega_sola_202208231608-coast3_hsi.zip](https://data.archive.sigma2.no/dataset/6b1f3cf3-6f88-4d6c-a5dc-79f03d6d51b3/download/vega/sola/aerial/hsi/20220823/massimal_vega_sola_202208231608-coast3_hsi/processed/massimal_vega_sola_202208231608-coast3_hsi.zip). 

To run this code on your own machine, create a virtual environment, install MassiPipe (and optionally rich), download the data ZIP file, and update the paths in the third cell below. 

In [1]:
# Imports
from pathlib import Path
from rich import print  # Not required, but makes printed configuration look nicer

import massipipe
import shutil

In [2]:
# Paths
example_hsi_dataset_path = Path(
    "/data/massimal/seabee-minio/vega/sola/aerial/hsi/20220823/"
    "massimal_vega_sola_202208231608-coast3_hsi"
)  # Path to dataset "root" - contains 1a_radiance forlder and YAML file

In [3]:
# Delete existing refletance images (only for demonstration purposes, not required in normal usage)
if (example_hsi_dataset_path / "2a_reflectance").exists():
    shutil.rmtree(example_hsi_dataset_path / "2a_reflectance")

In [4]:
# Show contents of the example dataset (requires tree to be installed - skip if not)
!tree {example_hsi_dataset_path} -L 2

/data/massimal/seabee-minio/vega/sola/aerial/hsi/20220823/massimal_vega_sola_202208231608-coast3_hsi
├── 1a_radiance
│   ├── massimal_vega_sola_202208231608-coast3_hsi_000_irradiance.spec
│   ├── massimal_vega_sola_202208231608-coast3_hsi_000_irradiance.spec.hdr
│   ├── massimal_vega_sola_202208231608-coast3_hsi_000_radiance.bip
│   ├── massimal_vega_sola_202208231608-coast3_hsi_000_radiance.bip.hdr
│   ├── massimal_vega_sola_202208231608-coast3_hsi_001_irradiance.spec
│   ├── massimal_vega_sola_202208231608-coast3_hsi_001_irradiance.spec.hdr
│   ├── massimal_vega_sola_202208231608-coast3_hsi_001_radiance.bip
│   ├── massimal_vega_sola_202208231608-coast3_hsi_001_radiance.bip.hdr
│   ├── massimal_vega_sola_202208231608-coast3_hsi_002_irradiance.spec
│   ├── massimal_vega_sola_202208231608-coast3_hsi_002_irradiance.spec.hdr
│   ├── massimal_vega_sola_202208231608-coast3_hsi_002_radiance.bip
│   ├── massimal_vega_sola_202208231608-coast3_hsi_002_radiance.bip.hdr
│   ├── massimal_vega_sol

In [5]:
# Create a pipeline for the dataset
pipeline = massipipe.pipeline.Pipeline(dataset_dir=example_hsi_dataset_path)

09:45:00 INFO: ----------------------------------------------------------------
09:45:00 INFO: File logging for massimal_vega_sola_202208231608-coast3_hsi initialized.
09:45:01 INFO: No raw files found, but radiance directory 1a_radiance found - processing based on radiance data.


In [6]:
# Print the configuration (read from YAML file)
print(pipeline.config)

MassipipeOptions(
    general=MpGeneral(rgb_wl=(640.0, 550.0, 460.0)),
    quicklook=MpQuickLook(create=True, overwrite=False, percentiles=None),
    imu_data=MpImuData(create=True, overwrite=False),
    geotransform=MpGeoTransform(
        create=True,
        overwrite=True,
        camera_opening_angle_deg=36.5,
        pitch_offset_deg=4.0,
        roll_offset_deg=0.0,
        altitude_offset_m=2.5,
        utm_x_offset_m=0.0,
        utm_y_offset_m=0.0,
        assume_square_pixels=True
    ),
    radiance=MpRadiance(
        create=True,
        overwrite=False,
        set_saturated_pixels_to_zero=True,
        add_envi_mapinfo_to_header=True,
        add_irradiance_to_header=True
    ),
    radiance_rgb=MpRadianceRgb(create=True, overwrite=False),
    radiance_gc=MpRadianceGc(
        create=True,
        overwrite=False,
        smooth_spectra=True,
        subtract_dark_spec=False,
        set_negative_values_to_zero=False,
        reference_image_numbers=[15, 16, 21],
        reference_image_ranges=[(0, 350, 0, 900), (450, 1650, 400, 900), (0, 2000, 0, 350)]
    ),
    radiance_gc_rgb=MpRadianceGcRgb(create=True, overwrite=False),
    irradiance=MpIrradiance(create=True, overwrite=False),
    reflectance=MpReflectance(
        create=True,
        overwrite=False,
        wl_min=400,
        wl_max=930,
        conv_irrad_with_gauss=True,
        fwhm_irrad_smooth=3.5,
        smooth_spectra=False,
        refl_from_mean_irrad=False
    ),
    reflectance_gc=MpReflectanceGc(create=True, overwrite=False, smooth_spectra=True, method='from_rad_gc'),
    reflectance_gc_rgb=MpReflectanceGcRgb(create=False, overwrite=False),
    mosaic=MpMosaic(
        overview_factors=[2, 4, 8, 16, 32],
        visualization_mosaic='radiance',
        radiance_rgb=MpMosaicRadiance(create=True, overwrite=False),
        radiance_gc_rgb=MpMosaicRadianceGc(create=False, overwrite=False),
        reflectance_gc_rgb=MpMosaicReflectanceGc(create=False, overwrite=False)
    )
)

All the configurations that have the keyword "create" correspond to a possible data product that the pipeline can generate. If we call `pipeline.run()`, it will attempt to create the data products where create = True. See the [MassiPipe documentation](https://mh-skjelvareid.github.io/massipipe/) for details. However, in this case we are only really looking to create reflectance images, and we call the function `convert_radiance_images_to_reflectance()` to do just that, and nothing more.

In [7]:
# Delete existing refletance images (only for demonstration purposes, not required in normal usage)
if (pipeline.dataset_dir / "2a_reflectance").exists():
    shutil.rmtree(pipeline.dataset_dir / "2a_reflectance")

# Convert radiance images to reflectance images
pipeline.convert_radiance_images_to_reflectance()

09:45:01 INFO: ---- REFLECTANCE CONVERSION ----
09:45:01 INFO: Converting massimal_vega_sola_202208231608-coast3_hsi_000_radiance.bip.hdr to reflectance.
09:45:12 INFO: Converting massimal_vega_sola_202208231608-coast3_hsi_001_radiance.bip.hdr to reflectance.
09:45:25 INFO: Converting massimal_vega_sola_202208231608-coast3_hsi_002_radiance.bip.hdr to reflectance.
09:45:40 INFO: Converting massimal_vega_sola_202208231608-coast3_hsi_003_radiance.bip.hdr to reflectance.
09:45:45 INFO: Converting massimal_vega_sola_202208231608-coast3_hsi_004_radiance.bip.hdr to reflectance.
09:45:55 INFO: Converting massimal_vega_sola_202208231608-coast3_hsi_005_radiance.bip.hdr to reflectance.
09:46:03 INFO: Converting massimal_vega_sola_202208231608-coast3_hsi_006_radiance.bip.hdr to reflectance.
09:46:09 INFO: Converting massimal_vega_sola_202208231608-coast3_hsi_007_radiance.bip.hdr to reflectance.
09:46:21 INFO: Converting massimal_vega_sola_202208231608-coast3_hsi_008_radiance.bip.hdr to reflectance

In [8]:
# Show file tree after creating reflectance images (requires tree to be installed - skip if not)
!tree {example_hsi_dataset_path} -L 2

/data/massimal/seabee-minio/vega/sola/aerial/hsi/20220823/massimal_vega_sola_202208231608-coast3_hsi
├── 1a_radiance
│   ├── massimal_vega_sola_202208231608-coast3_hsi_000_irradiance.spec
│   ├── massimal_vega_sola_202208231608-coast3_hsi_000_irradiance.spec.hdr
│   ├── massimal_vega_sola_202208231608-coast3_hsi_000_radiance.bip
│   ├── massimal_vega_sola_202208231608-coast3_hsi_000_radiance.bip.hdr
│   ├── massimal_vega_sola_202208231608-coast3_hsi_001_irradiance.spec
│   ├── massimal_vega_sola_202208231608-coast3_hsi_001_irradiance.spec.hdr
│   ├── massimal_vega_sola_202208231608-coast3_hsi_001_radiance.bip
│   ├── massimal_vega_sola_202208231608-coast3_hsi_001_radiance.bip.hdr
│   ├── massimal_vega_sola_202208231608-coast3_hsi_002_irradiance.spec
│   ├── massimal_vega_sola_202208231608-coast3_hsi_002_irradiance.spec.hdr
│   ├── massimal_vega_sola_202208231608-coast3_hsi_002_radiance.bip
│   ├── massimal_vega_sola_202208231608-coast3_hsi_002_radiance.bip.hdr
│   ├── massimal_vega_sol